# Final text model: PhoBERT-base-v2 clickbait classifier (Kaggle)

**Before running**: Accelerator **GPU**, Internet **On**. Code and processed data are cloned from GitHub (push your latest changes first).

Trains **one** model with the configuration chosen by the cross-validation experiments (out-of-fold F1 0.771):

| setting | value |
|---|---|
| input | title + lead paragraph, word-segmented, max 128 tokens |
| loss | class-weighted cross-entropy |
| optimiser | AdamW, lr 2e-5, layer-wise decay 0.9, 10% warm-up |
| training | 5 epochs, batch 32, best checkpoint on validation loss |

Trained on the train split, checkpoint chosen on val, scored once on the untouched test split. The saved folder (`best.pt`, `tokenizer/`, `config.json`) is what the extension's back end will load.

In [ ]:
!pip install -q pyvi

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Zaidve/TMG_project_re.git"
CLONE_DIR = Path("/tmp/TMG_project_re")  # outside /kaggle/working so it is not saved as output
ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_ROOTS = [Path("/kaggle/input"), CLONE_DIR]

def find(pattern):
    return [hit for root in SEARCH_ROOTS if root.exists() for hit in sorted(root.rglob(pattern))]

def find_parent(pattern, levels):
    """Directory `levels` above the first matching file in an attached dataset or the clone."""
    hits = find(pattern)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {[str(r) for r in SEARCH_ROOTS]}")
    return hits[0].parents[levels]

if ON_KAGGLE and not find("trainer/train_phobert.py"):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}

if ON_KAGGLE:
    CODE_DIR = find_parent("trainer/train_phobert.py", 1)
    DATA_DIR = find_parent("train.jsonl", 0)
    OUT_DIR = Path("/kaggle/working/phobert_final")
else:  # local run from the notebooks/ folder
    CODE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = CODE_DIR / "data" / "processed"
    OUT_DIR = CODE_DIR / "outputs" / "phobert_final"

sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out: ", OUT_DIR)

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Train

In [ ]:
from dataclasses import asdict
from trainer.train_phobert import Config, run

cfg = Config(data_dir=str(DATA_DIR), out_dir=str(OUT_DIR), num_workers=2 if ON_KAGGLE else 0)
print({k: v for k, v in asdict(cfg).items() if k not in ("data_dir", "out_dir")})
result = run(cfg)

## Test results

In [ ]:
import pandas as pd

history = pd.DataFrame(result["history"]).set_index("epoch")
display(history[["train_loss", "val_loss", "val_f1", "val_precision", "val_recall"]].round(4))
pd.Series(result["test_metrics"]).round(4).to_frame("test")

In [ ]:
preds = result["test_predictions"]
pd.crosstab(preds["label"], preds["pred"], rownames=["actual"], colnames=["predicted"], margins=True)

## Load the saved model and score new headlines

`ClickbaitPredictor` applies the same cleaning and word segmentation as training, so raw text from a web page can go straight in.

In [ ]:
from utils.predict import ClickbaitPredictor

predictor = ClickbaitPredictor.from_dir(OUT_DIR)
examples = [
    "Sốc: 5 bí quyết này ai cũng nên biết!",
    "Vì sao giá vàng thế giới được dự báo giảm mạnh?",
    "Quy hoạch 9 khu chức năng phía đông TP HCM",
    "Sân bay Vinh đóng cửa 6 tháng để nâng cấp",
]
pd.DataFrame(predictor.predict(examples), index=examples)

## Package for download

Zips the model folder (about 0.5 GB) into `/kaggle/working/phobert_final.zip`; download it from the notebook's **Output** tab after **Save Version**.

In [ ]:
import os
import shutil

archive = shutil.make_archive(str(OUT_DIR), "zip", root_dir=OUT_DIR)
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT_DIR)}")
print(f"\n{archive}: {os.path.getsize(archive) / 1e6:.0f} MB")